# M2b · Cleaning-rule exploration

**Goal:** for each of C1–C7, count how many rows would be **rejected**, **clamped**, or **nullified** if we applied the rule to staging today.

No writes. The cleaning rules live in `config/cleaning_rules.yaml` and the engine in `src/accent_fleet/cleaning/rules.py`.

**Exit criterion:** you can state rejection % per rule per source table.


In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs

In [ ]:
from accent_fleet.config import load_cleaning_rules
rules = load_cleaning_rules()
import json
print(json.dumps(rules, indent=2, default=str)[:1500], '...')


## 3. Execute — count would-be rejects per rule directly in SQL

In [ ]:
# We translate each rule's SQL condition into a COUNT probe. The engine itself
# runs per-row in Python/Polars; here we just want aggregate counts for speed.
import pandas as pd
probes = []
for r in rules.get('rules', []):
    rid = r['id']; tbl = r['target_table']; cond = r['condition']; action = r['action']
    # `condition` is a SQL predicate that is TRUE when the rule matches.
    q = text(f"SELECT COUNT(*) FROM staging.{tbl} WHERE {cond}")
    try:
        with get_engine().connect() as conn:
            n = conn.execute(q).scalar_one()
        probes.append({'rule': rid, 'action': action, 'table': tbl, 'matched': n})
    except Exception as exc:
        probes.append({'rule': rid, 'action': action, 'table': tbl, 'matched': f'ERR: {exc}'})
pd.DataFrame(probes)


## 4. Inspect

In [ ]:
# Compare to total rows per table so you can talk in percentages.
with get_engine().connect() as conn:
    totals = {t['target_table']: conn.execute(text(f"SELECT COUNT(*) FROM staging.{t['target_table']}")).scalar_one()
              for t in rules.get('rules', [])}
import pandas as pd
df = pd.DataFrame(probes)
df['total'] = df['table'].map(totals)
df['pct'] = (df['matched'].where(df['matched'].apply(lambda v: isinstance(v, (int,float))), 0) / df['total'] * 100).round(3)
df
